# Prep route and stop gdf
1. only subset for operators by region
2. do aggregation?
3. subset to special routes
   if this is skipped, after step 2, can go straight to regression?

In [1]:
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import C4_event_helpers as C4
import E1_new_prep as E1

credentials, _ = google.auth.default()

In [3]:
route_gdf = E1.prep_fct_daily_schedule_rt_route_direction_summary(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    event_time_of_day_dict = wc_vars.sofi_match_times,
) 

In [6]:
trips_by_event = C4.aggregate_by_event_type(
    route_gdf,
    group_cols=[
        "schedule_name",
        "route_name",
        "direction_id",
        "event_day",
        "day_type",
    ],
    metric_cols=["n_trips"],
).rename(columns={"n_trips": "daily_trips"})

In [7]:
trips_wide = C4.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

In [10]:
import importlib
importlib.reload(C4)

<module 'C4_event_helpers' from '/home/jovyan/gtfs-curator/transit_events/C4_event_helpers.py'>

In [11]:
trips_wide_gdf = C4.merge_routes_with_shape_geom(trips_wide)   

ArrowInvalid: No match for FieldRef.Name(schedule_name) in feed_key: string
shape_array_key: string
shape_id: string
geometry: binary
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string

In [ ]:
trips_wide_gdf.head(2)

In [ ]:
trips_wide_gdf_near = E1.categorize_proximity_to_poi(trips_wide_gdf, stadium_gdf)

In [ ]:
route_gdf.schedule_name.value_counts()

In [ ]:
sofi_trips = D1.filter_to_special_routes(
    route_gdf,
    route_name_dict = wc_vars.special_socal_routes_dict
)

In [ ]:
trips_by_event = D2.aggregate_by_event_type(
    sofi_trips, 
    group_cols = ["schedule_name", "route_name", "direction_id", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

In [ ]:
trips_by_event2 = D2.aggregate_by_event_type(
    route_gdf, 
    group_cols = ["schedule_name", "route_name", "direction_id", "event_day", "day_type"], 
    metric_cols = ["n_trips"]
).rename(columns = {
    "n_trips": "daily_trips", 
})

trips_wide2 = D2.make_wide(
    trips_by_event,
    index_cols=["schedule_name", "route_name", "direction_id"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_trips"],
)

In [ ]:
# on the full df, this makes sense, most of them are 0s
trips_wide2.change_daily_trips_weekday.value_counts(normalize=True)